# QWZ Model Simulation

This notebook works through the Qi-Wu-Zhang 2-band model. The aim is to set up the canonical form of the model and confirm its topology using the standard berry curvature method and spectral localiser (which requires a real-space expression). After this, additional terms can be introduced to alter the bandgap, creating an indirect bandgap. Now, fractional Chern numbers can be calcualted by noth methods. Thereafter, disorder of various kinds can be added and tuned to investigate the effect on topology in the direct and indirect bandgap limits.



In [173]:
using LinearAlgebra
using Printf
using Plots
using DataFrames

In [174]:
## Pauli matrices
sigma_x = [0 1; 1 0]
sigma_y = [0 -im; im 0]
sigma_z = [1 0; 0 -1]
identity = [1 0; 0 1]

2×2 Matrix{Int64}:
 1  0
 0  1

## Hamiltonian (k-space)

Let:
$$\vec\sigma = (\sigma_x, \sigma_y, \sigma_z)$$
$$\vec d(\mathbf{k}) = (d_x(\mathbf{k}), d_y(\mathbf{k}), d_z(\mathbf{k}))$$

where
$$d_x(\mathbf{k}) = A\sin{(k_x})$$
$$d_y(\mathbf{k}) = A\sin{(k_y})$$
$$d_z(\mathbf{k}) = m + B(2 - \cos{(k_x)}-\cos{(k_y)})$$
\begin{equation}
\begin{split}
&\text{set} \ B=1 \\
&= (m+2) - \cos{(k_x)}-\cos{(k_y)} \\
&= M - \cos{(k_x)}-\cos{(k_y)}
\end{split}
\end{equation}

The Hamiltonian is
$$H(\mathbf{k}) = d_x(\mathbf{k})\sigma_x + d_y(\mathbf{k})\sigma_y + d_z(\mathbf{k})\sigma_z$$

Which can be solved by taking $H(\mathbf{k})^2=|\vec d|^2\mathbf{I}$ to give (N.B. taking $A=B=1$)
$$E_\pm(\mathbf{k}) = \pm|\vec d(\mathbf{k})| = \pm\sqrt{\sin^2{(k_x)} + \sin^2{(k_y)} + (M-\cos(k_x) - \cos{(k_y)})^2}$$


In [175]:
# QWZ Hamiltonian with general A, B, m
function hamiltonian_qwz(
    kx::Real,
    ky::Real;
    A::Real=1.0,
    B::Real=1.0,
    m::Real=0.0
    )::Matrix{ComplexF64}

    dx::Float64 = Float64(A) * sin(Float64(kx))
    dy::Float64 = Float64(A) * sin(Float64(ky))
    dz::Float64 = Float64(m) + Float64(B) * (2.0 - cos(Float64(kx)) - cos(Float64(ky)))
    return ComplexF64.(dx * sigma_x + dy * sigma_y + dz * sigma_z)
end

# Analytic QWZ band energies E_±(k) = ±sqrt(dx^2 + dy^2 + dz^2)
function band_energies_qwz(
    kx::Real,
    ky::Real;
    A::Real=1.0,
    B::Real=1.0,
    m::Real=0.0
    )::NTuple{2, Float64}

    dx::Float64 = Float64(A) * sin(Float64(kx))
    dy::Float64 = Float64(A) * sin(Float64(ky))
    dz::Float64 = Float64(m) + Float64(B) * (2.0 - cos(Float64(kx)) - cos(Float64(ky)))
    e::Float64 = sqrt(dx * dx + dy * dy + dz * dz)
    return (-e, e)
end

function band_structure_qwz(
    A::Real=1.0,
    B::Real=1.0,
    m::Real=0.0;
    nk::Int=51
    )::Tuple{Vector{Float64}, Vector{Float64}, Array{Float64, 3}}

    ks::Vector{Float64} = collect(range(-pi, pi; length=nk + 1))[1:end-1]  # avoid duplicate endpoint at +pi
    Evals::Array{Float64, 3} = zeros(Float64, length(ks), length(ks), 2)

    for (i, kx) in enumerate(ks), (j, ky) in enumerate(ks)
        e1, e2 = band_energies_qwz(kx, ky; A=A, B=B, m=m)
        Evals[i, j, 1] = e1
        Evals[i, j, 2] = e2
    end

    return ks, ks, Evals
end

# Wide-format table: one row per (A, B, m, kx, ky), with both bands in separate columns.
function build_evals_df(
    Avals::AbstractVector{<:Real},
    Bvals::AbstractVector{<:Real},
    mvals::AbstractVector{<:Real};
    nk::Int=51
    )::DataFrame

    ks::Vector{Float64} = collect(range(-pi, pi; length=nk))[1:end-1]  # avoid duplicate endpoint at +pi
    nrows::Int = length(Avals) * length(Bvals) * length(mvals) * length(ks)^2

    Acol = Vector{Float64}(undef, nrows)
    Bcol = Vector{Float64}(undef, nrows)
    mcol = Vector{Float64}(undef, nrows)
    kxcol = Vector{Float64}(undef, nrows)
    kycol = Vector{Float64}(undef, nrows)
    Eminus_col = Vector{Float64}(undef, nrows)
    Eplus_col = Vector{Float64}(undef, nrows)

    idx::Int = 1
    for A in Avals, B in Bvals, m in mvals, kx in ks, ky in ks
        e1, e2 = band_energies_qwz(kx, ky; A=A, B=B, m=m)
        @inbounds begin
            Acol[idx] = Float64(A); Bcol[idx] = Float64(B); mcol[idx] = Float64(m)
            kxcol[idx] = kx; kycol[idx] = ky
            Eminus_col[idx] = e1; Eplus_col[idx] = e2
            idx += 1
        end
    end

    return DataFrame(
        A=Acol,
        B=Bcol,
        m=mcol,
        kx=kxcol,
        ky=kycol,
        E_minus=Eminus_col,
        E_plus=Eplus_col
    )
end

build_evals_df (generic function with 1 method)

In [176]:
Avals = [1.0]
Bvals = [1.0]
mvals = collect(-4.0:1:4.0) #[-2.0, 0.0, 2.0]
nk = 501

# qwz_df = build_evals_df(Avals, Bvals, mvals; nk=nk)
println("Built DataFrame with $(nrow(qwz_df)) rows and $(ncol(qwz_df)) columns.")

Built DataFrame with 2250000 rows and 7 columns.


In [177]:
## plots the qwz bandstructure for the E+ and E- bands separately
function plt_qwz_bandstructure_split(
    df::DataFrame;
    atol::Real=1e-8,
    fixed_variables...
    )
    # Verify required columns exist (using string comparison for robustness)
    required_cols = ["kx", "ky", "E_minus", "E_plus"]
    col_names_str = string.(names(df))
    for c in required_cols
        c in col_names_str || error("Missing required column: $c")
    end

    subdf = df
    for (k, v) in fixed_variables
        string(k) in col_names_str || error("Filter key $(k) is not a DataFrame column.")
        col = df[!, k]
        if v isa Real && eltype(col) <: Real
            mask = abs.(Float64.(col) .- Float64(v)) .<= Float64(atol)
            subdf = subdf[mask, :]
        else
            mask = col .== v
            subdf = subdf[mask, :]
        end
    end

    nrow(subdf) > 0 || error("No rows matched the requested fixed variables. Try increasing atol or checking available parameter values.")

    kx_vals = sort(unique(subdf.kx))
    ky_vals = sort(unique(subdf.ky))
    nx = length(kx_vals)
    ny = length(ky_vals)

    ix = Dict(kx_vals[i] => i for i in eachindex(kx_vals))
    iy = Dict(ky_vals[j] => j for j in eachindex(ky_vals))

    Emin = fill(NaN, nx, ny)
    Emax = fill(NaN, nx, ny)

    for row in eachrow(subdf)
        i = ix[row.kx]
        j = iy[row.ky]
        Emin[i, j] = row.E_minus
        Emax[i, j] = row.E_plus
    end

    p1 = surface(
        kx_vals, ky_vals, Emin';
        xlabel="k_x", ylabel="k_y", zlabel="E",
        title="Lower band E_-",
        color=:viridis,
        legend=false
    )
    p2 = surface(
        kx_vals, ky_vals, Emax';
        xlabel="k_x", ylabel="k_y", zlabel="E",
        title="Upper band E_+",
        color=:plasma,
        legend=false
    )

    label_text = isempty(fixed_variables) ? "All rows" : join(["$(k)=$(v)" for (k, v) in fixed_variables], ", ")
    plt = plot(p1, p2; layout=(1, 2), size=(1200, 460), plot_title="QWZ Band Structure " * label_text)
    display(plt)
    # return plt, subdf
end

## plots the qwz bandstructure for the E+ and E- bands together in one plot
function plt_qwz_bandstructure_combined(
    df::DataFrame;
    atol::Real=1e-8,
    fixed_variables...
    )
    # Verify required columns exist (using string comparison for robustness)
    required_cols = ["kx", "ky", "E_minus", "E_plus"]
    col_names_str = string.(names(df))
    for c in required_cols
        c in col_names_str || error("Missing required column: $c")
    end

    subdf = df
    for (k, v) in fixed_variables
        string(k) in col_names_str || error("Filter key $(k) is not a DataFrame column.")
        col = df[!, k]
        if v isa Real && eltype(col) <: Real
            mask = abs.(Float64.(col) .- Float64(v)) .<= Float64(atol)
            subdf = subdf[mask, :]
        else
            mask = col .== v
            subdf = subdf[mask, :]
        end
    end

    nrow(subdf) > 0 || error("No rows matched the requested fixed variables. Try increasing atol or checking available parameter values.")

    kx_vals = sort(unique(subdf.kx))
    ky_vals = sort(unique(subdf.ky))
    nx = length(kx_vals)
    ny = length(ky_vals)

    ix = Dict(kx_vals[i] => i for i in eachindex(kx_vals))
    iy = Dict(ky_vals[j] => j for j in eachindex(ky_vals))

    Emin = fill(NaN, nx, ny)
    Emax = fill(NaN, nx, ny)

    for row in eachrow(subdf)
        i = ix[row.kx]
        j = iy[row.ky]
        Emin[i, j] = row.E_minus
        Emax[i, j] = row.E_plus
    end

    # Combined plot: both bands overlaid on the same 3D surface
    plt = surface(
        kx_vals, ky_vals, Emin';
        xlabel="k_x", ylabel="k_y", zlabel="E",
        color=:viridis,
        # alpha=0.75,
        legend=false
    )

    surface!(
        plt,
        kx_vals, ky_vals, Emax';
        color=:plasma,
        # alpha=0.75,
        legend=false,
    )

    label_text = isempty(fixed_variables) ? "All rows" : join(["$(k)=$(v)" for (k, v) in fixed_variables], ", ")
    plot!(plt; plot_title="QWZ Band Structure (Combined) " * label_text, size=(1000, 700))
    display(plt)
    # return plt, subdf
end

## plots the qwz bandstructure for a controllable slice in kx or ky (E+ and E- combined in same plot)
function plt_qwz_bandstructure_slice(
    df::DataFrame;
    atol::Real=1e-8,
    fixed_variables...
    )
    # Verify required columns exist (using string comparison for robustness)
    required_cols = ["kx", "ky", "E_minus", "E_plus"]
    col_names_str = string.(names(df))
    for c in required_cols
        c in col_names_str || error("Missing required column: $c")
    end

    subdf = df
    for (k, v) in fixed_variables
        string(k) in col_names_str || error("Filter key $(k) is not a DataFrame column.")
        col = subdf[!, k]
        if v isa Real && eltype(col) <: Real
            col_vals = Float64.(col)
            chosen = col_vals[argmin(abs.(col_vals .- Float64(v)))]
            if abs(chosen - Float64(v)) > Float64(atol)
                @warn "Snapping $(k)=$(v) to nearest available grid value $(chosen)"
            end
            subdf = subdf[col_vals .== chosen, :]
        else
            mask = col .== v
            subdf = subdf[mask, :]
        end
    end

    nrow(subdf) > 0 || error("No rows matched the requested fixed variables. Try increasing atol or checking available parameter values.")

    fixed_map = Dict(fixed_variables)
    kx_fixed = haskey(fixed_map, :kx)
    ky_fixed = haskey(fixed_map, :ky)
    (kx_fixed ⊻ ky_fixed) || error("Specify exactly one of kx or ky as the fixed momentum coordinate.")

    fixed_axis = kx_fixed ? :kx : :ky
    free_axis = kx_fixed ? :ky : :kx

    axis_vals = sort(unique(subdf[!, free_axis]))

    Eminus = similar(axis_vals, Float64)
    Eplus = similar(axis_vals, Float64)

    for (idx, val) in pairs(axis_vals)
        row = subdf[subdf[!, free_axis] .== val, :]
        nrow(row) == 1 || error("Expected exactly one row for $(free_axis)=$(val), but found $(nrow(row)).")
        Eminus[idx] = row.E_minus[1]
        Eplus[idx] = row.E_plus[1]
    end

    fixed_desc = join(["$(k)=$(v)" for (k, v) in fixed_map], ", ")
    xlabel_text = String(free_axis)
    title_text = "QWZ slice: fixed $(fixed_axis) | $(fixed_desc)"

    plt = plot(
        axis_vals, Eminus;
        xlabel=xlabel_text,
        ylabel="E",
        label="E_-",
        color=:dodgerblue3,
        linewidth=2.5,
        title=title_text,
        legend=:topright,
        ylim=(-4.1, 4.1),
    )
    plot!(
        plt,
        axis_vals, Eplus;
        label="E_+",
        color=:crimson,
        linewidth=2.5
    )

    hline!(plt, [0.0]; linestyle=:dash, color=:black, linewidth=1, label=false)
    display(plt)
    # return plt, subdf
end

plt_qwz_bandstructure_slice (generic function with 1 method)

In [178]:
# plt_qwz_bandstructure_split(qwz_df; m=0.0)

In [179]:
# plt_qwz_bandstructure_combined(qwz_df; m=0.0)

In [180]:
# plt_qwz_bandstructure_slice(qwz_df; m=-4.0, kx=pi)